# 🥬 Notebook 2: Klasik Makine Öğrenmesi ile Sınıflandırma

Bu notebook'ta Notebook 1'de çıkarılan el yapımı özellikler kullanılarak çeşitli klasik makine öğrenmesi modelleri eğitilmektedir.

**İçerik:**
- Özellik matrisini yükleme ve hazırlama
- Train/Validation/Test bölme (70/15/15)
- Boyut indirgeme (PCA, LDA)
- 10 farklı ML modeli (SVM, RF, XGBoost, LightGBM, CatBoost, KNN, LR, NB)
- Optuna ile hiperparametre optimizasyonu
- Stratified K-Fold Cross Validation
- Model karşılaştırma ve analiz

In [ ]:
# Kütüphane İmportları
import os
import warnings
warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib

# Makine öğrenmesi
from sklearn.model_selection import (train_test_split, StratifiedKFold, 
                                      cross_val_score, GridSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest, chi2
from sklearn.metrics import (classification_report, confusion_matrix, 
                              roc_auc_score, accuracy_score, f1_score)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Hiperparametre optimizasyonu
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✅ Tüm kütüphaneler başarıyla yüklendi!")

## ⚙️ Konfigürasyon

In [ ]:
# Konfigürasyon
RESULTS_DIR = "./results/"
FEATURES_PATH = "../01_eda_feature_engineering/results/features.csv"
RANDOM_SEED = 42
N_FOLDS = 5
TEST_SIZE = 0.15
VAL_SIZE = 0.15

os.makedirs(RESULTS_DIR, exist_ok=True)

import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"📁 Sonuç dizini: {RESULTS_DIR}")
print(f"📊 Özellik dosyası: {FEATURES_PATH}")

## 📂 Veri Yükleme

Notebook 1'de çıkarılan özellik matrisi yüklenmektedir.

In [ ]:
# Özellik matrisini yükle
if os.path.exists(FEATURES_PATH):
    feature_df = pd.read_csv(FEATURES_PATH)
    print(f"✅ Özellik matrisi yüklendi: {feature_df.shape}")
else:
    print("⚠️  Özellik dosyası bulunamadı. Demo veri oluşturuluyor...")
    # Demo veri oluştur
    np.random.seed(RANDOM_SEED)
    n_samples = 200
    n_features = 500
    X_demo = np.random.randn(n_samples, n_features)
    labels_demo = np.repeat(np.arange(1, 24), n_samples // 23 + 1)[:n_samples]
    feature_df = pd.DataFrame(X_demo, columns=[f'feat_{i}' for i in range(n_features)])
    feature_df['label'] = labels_demo.astype(str)
    print(f"✅ Demo veri oluşturuldu: {feature_df.shape}")

# X ve y ayır
X = feature_df.drop('label', axis=1).values
y = feature_df['label'].values

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n📊 Özellik matrisi: {X.shape}")
print(f"🏷️  Sınıf sayısı: {len(np.unique(y))}")
print(f"📋 Sınıflar: {le.classes_}")

## 🔀 Train/Validation/Test Bölme (70/15/15)

Veri seti stratified olarak bölünmektedir.

In [ ]:
# Stratified split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=RANDOM_SEED, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"📊 Veri bölme sonuçları:")
print(f"  Train: {X_train.shape[0]} örnek ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Val:   {X_val.shape[0]} örnek ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"  Test:  {X_test.shape[0]} örnek ({X_test.shape[0]/len(X)*100:.1f}%)")

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\n✅ StandardScaler uygulandı.")

## 📉 Boyut İndirgeme Analizi

### PCA (Principal Component Analysis)

PCA ile açıklanan varyans oranı analiz edilmektedir.

In [ ]:
# PCA analizi
pca_full = PCA(random_state=RANDOM_SEED)
pca_full.fit(X_train_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumvar >= 0.95) + 1
n_99 = np.argmax(cumvar >= 0.99) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cumvar, linewidth=2, color='steelblue')
axes[0].axhline(0.95, color='red', linestyle='--', label=f'%95 = {n_95} bileşen')
axes[0].axhline(0.99, color='orange', linestyle='--', label=f'%99 = {n_99} bileşen')
axes[0].set_xlabel('Bileşen Sayısı', fontsize=12)
axes[0].set_ylabel('Kümülatif Açıklanan Varyans', fontsize=12)
axes[0].set_title('PCA: Kümülatif Açıklanan Varyans', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(1, min(51, len(pca_full.explained_variance_ratio_)+1)),
            pca_full.explained_variance_ratio_[:50], color='steelblue', alpha=0.7)
axes[1].set_xlabel('Bileşen', fontsize=12)
axes[1].set_ylabel('Açıklanan Varyans Oranı', fontsize=12)
axes[1].set_title('İlk 50 PCA Bileşeni', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pca_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ PCA Sonuçları:")
print(f"  %95 varyans için: {n_95} bileşen")
print(f"  %99 varyans için: {n_99} bileşen")

# PCA uygula
n_components = min(n_95, X_train_scaled.shape[1], X_train_scaled.shape[0]-1)
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"\n✅ PCA uygulandı: {X_train_pca.shape[1]} bileşen")

In [ ]:
# LDA uygula
n_lda = min(len(np.unique(y_encoded)) - 1, X_train_scaled.shape[1])
lda = LDA(n_components=n_lda)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_val_lda = lda.transform(X_val_scaled)
X_test_lda = lda.transform(X_test_scaled)
print(f"✅ LDA uygulandı: {X_train_lda.shape[1]} bileşen")
print(f"  LDA açıklanan varyans: {lda.explained_variance_ratio_[:5].sum():.3f} (ilk 5 bileşen)")

## 🤖 Model Eğitimi

### 1. SVM (Support Vector Machine)

In [ ]:
# Yardımcı fonksiyonlar
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name, cv=N_FOLDS):
    """Modeli değerlendirir ve sonuçları döndürür."""
    start = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - start
    
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    macro_f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    
    # Cross validation
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy')
    
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"  Accuracy:      {acc:.4f}")
    print(f"  Macro F1:      {macro_f1:.4f}")
    print(f"  Weighted F1:   {weighted_f1:.4f}")
    print(f"  CV Accuracy:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Eğitim süresi: {train_time:.2f}s")
    
    return {
        'model_name': model_name,
        'model': model,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'train_time': train_time,
        'y_pred': y_pred
    }

results = []

In [ ]:
# SVM - RBF Kernel (GridSearchCV)
print("🔧 SVM - RBF Kernel eğitiliyor...")
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
svm_rbf = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=RANDOM_SEED),
                        param_grid, cv=3, scoring='accuracy', n_jobs=-1)
res = evaluate_model(svm_rbf, X_train_pca, y_train, X_val_pca, y_val, 'SVM-RBF')
results.append(res)
print(f"  En iyi params: {svm_rbf.best_params_}")

In [ ]:
# SVM - Polynomial Kernel
print("🔧 SVM - Polynomial Kernel eğitiliyor...")
svm_poly = SVC(kernel='poly', degree=3, probability=True, C=1.0, random_state=RANDOM_SEED)
res = evaluate_model(svm_poly, X_train_pca, y_train, X_val_pca, y_val, 'SVM-Poly')
results.append(res)

In [ ]:
# SVM - Linear Kernel
print("🔧 SVM - Linear Kernel eğitiliyor...")
svm_linear = SVC(kernel='linear', probability=True, C=1.0, random_state=RANDOM_SEED)
res = evaluate_model(svm_linear, X_train_pca, y_train, X_val_pca, y_val, 'SVM-Linear')
results.append(res)

In [ ]:
# Random Forest
print("🔧 Random Forest eğitiliyor...")
rf = RandomForestClassifier(n_estimators=200, max_depth=None, 
                             random_state=RANDOM_SEED, n_jobs=-1)
res = evaluate_model(rf, X_train_scaled, y_train, X_val_scaled, y_val, 'Random Forest')
results.append(res)

# Feature Importance
feat_imp = pd.Series(rf.feature_importances_).nlargest(20)
fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest - Top 20 Özellik Önemi', fontsize=13, fontweight='bold')
ax.set_xlabel('Önem Skoru')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost with Optuna
print("🔧 XGBoost (Optuna) eğitiliyor...")

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss'
    }
    model = XGBClassifier(**params)
    score = cross_val_score(model, X_train_pca, y_train, cv=3, scoring='accuracy').mean()
    return score

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=20, show_progress_bar=False)
best_xgb = XGBClassifier(**study_xgb.best_params, random_state=RANDOM_SEED, 
                           n_jobs=-1, eval_metric='mlogloss')
res = evaluate_model(best_xgb, X_train_pca, y_train, X_val_pca, y_val, 'XGBoost')
results.append(res)
print(f"  En iyi params: {study_xgb.best_params}")

In [ ]:
# LightGBM with Optuna
print("🔧 LightGBM (Optuna) eğitiliyor...")

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    score = cross_val_score(model, X_train_pca, y_train, cv=3, scoring='accuracy').mean()
    return score

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lgbm_objective, n_trials=20, show_progress_bar=False)
best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=RANDOM_SEED, 
                             n_jobs=-1, verbose=-1)
res = evaluate_model(best_lgbm, X_train_pca, y_train, X_val_pca, y_val, 'LightGBM')
results.append(res)

In [ ]:
# CatBoost
print("🔧 CatBoost eğitiliyor...")
catboost = CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1,
                               random_seed=RANDOM_SEED, verbose=0)
res = evaluate_model(catboost, X_train_pca, y_train, X_val_pca, y_val, 'CatBoost')
results.append(res)

In [ ]:
# KNN - farklı k değerleri
print("🔧 KNN eğitiliyor...")
best_knn_acc = 0
best_k = 5

for k in [3, 5, 7, 9, 11]:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train_pca, y_train)
    acc = accuracy_score(y_val, knn.predict(X_val_pca))
    print(f"  k={k}: Accuracy = {acc:.4f}")
    if acc > best_knn_acc:
        best_knn_acc = acc
        best_k = k

best_knn = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
res = evaluate_model(best_knn, X_train_pca, y_train, X_val_pca, y_val, f'KNN (k={best_k})')
results.append(res)

In [ ]:
# Logistic Regression (One-vs-Rest)
print("🔧 Logistic Regression eğitiliyor...")
lr = LogisticRegression(multi_class='ovr', max_iter=1000, 
                         C=1.0, random_state=RANDOM_SEED, n_jobs=-1)
res = evaluate_model(lr, X_train_pca, y_train, X_val_pca, y_val, 'Logistic Regression')
results.append(res)

In [ ]:
# Gaussian Naive Bayes
print("🔧 Gaussian Naive Bayes eğitiliyor...")
gnb = GaussianNB()
res = evaluate_model(gnb, X_train_pca, y_train, X_val_pca, y_val, 'Gaussian NB')
results.append(res)

## 📊 Model Karşılaştırma

In [ ]:
# Sonuçları DataFrame'e dönüştür
results_df = pd.DataFrame([{
    'Model': r['model_name'],
    'Val Accuracy': r['accuracy'],
    'Macro F1': r['macro_f1'],
    'Weighted F1': r['weighted_f1'],
    'CV Mean': r['cv_mean'],
    'CV Std': r['cv_std'],
    'Train Time (s)': round(r['train_time'], 2)
} for r in results]).sort_values('Val Accuracy', ascending=False)

print("\n📊 Model Karşılaştırma Tablosu:")
print(results_df.to_string(index=False))

# Sonuçları kaydet
results_df.to_csv(os.path.join(RESULTS_DIR, 'model_comparison.csv'), index=False)

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['Val Accuracy', 'Macro F1', 'Weighted F1']
colors = ['steelblue', 'seagreen', 'darkorange']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df.sort_values(metric, ascending=True)
    ax.barh(sorted_df['Model'], sorted_df[metric], color=color, alpha=0.8)
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(f'Model Karşılaştırma - {metric}', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1)
    for i, v in enumerate(sorted_df[metric]):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🔍 En İyi Model Detaylı Analizi

Confusion matrix ve classification report görüntülenmektedir.

In [ ]:
# En iyi model
best_result = max(results, key=lambda x: x['accuracy'])
best_model = best_result['model']
best_name = best_result['model_name']

print(f"🏆 En iyi model: {best_name}")
print(f"   Accuracy: {best_result['accuracy']:.4f}")

# Test seti değerlendirmesi
if 'pca' in best_name.lower() or best_name in ['SVM-RBF', 'SVM-Poly', 'SVM-Linear', 
                                                  'XGBoost', 'LightGBM', 'CatBoost',
                                                  'KNN (k=5)', 'Logistic Regression']:
    X_test_eval = X_test_pca
else:
    X_test_eval = X_test_scaled

y_test_pred = best_model.predict(X_test_eval)
test_acc = accuracy_score(y_test, y_test_pred)
print(f"\n📊 TEST Seti Sonuçları ({best_name}):")
print(f"   Test Accuracy: {test_acc:.4f}")
print(f"\n{classification_report(y_test, y_test_pred, target_names=le.classes_)}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=14, fontweight='bold')
ax.set_ylabel('Gerçek Etiket', fontsize=12)
ax.set_xlabel('Tahmin Edilen Etiket', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'confusion_matrix_{best_name.replace(" ", "_")}.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

## 💾 Model ve Tahmin Kaydetme

En iyi modeller kaydedilmekte ve ensemble için tahmin olasılıkları saklanmaktadır.

In [ ]:
# En iyi 3 modeli kaydet
top3_results = sorted(results, key=lambda x: x['accuracy'], reverse=True)[:3]

for i, res in enumerate(top3_results):
    model_path = os.path.join(RESULTS_DIR, f'model_top{i+1}_{res["model_name"].replace(" ", "_")}.pkl')
    joblib.dump(res['model'], model_path)
    print(f"✅ Model kaydedildi: {model_path}")

# Tahmin olasılıklarını kaydet (ensemble için)
for res in results:
    model = res['model']
    model_name = res['model_name'].replace(' ', '_').replace('(', '').replace(')', '')
    
    try:
        if hasattr(model, 'predict_proba'):
            # Hangi X setinin kullanıldığını belirle
            X_eval = X_test_pca if hasattr(model, 'best_estimator_') or 'SVM' in res['model_name'] else X_test_scaled
            proba = model.predict_proba(X_eval)
            proba_path = os.path.join(RESULTS_DIR, f'proba_{model_name}.npy')
            np.save(proba_path, proba)
            print(f"✅ Olasılıklar kaydedildi: {proba_path} - shape: {proba.shape}")
    except Exception as e:
        print(f"⚠️  {model_name} için olasılık kaydedilemedi: {e}")

# Etiketleri de kaydet
np.save(os.path.join(RESULTS_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(RESULTS_DIR, 'label_encoder_classes.npy'), le.classes_)
print("\n✅ Test etiketleri ve sınıf isimleri kaydedildi.")

## ✅ Özet

Bu notebook'ta aşağıdaki işlemler tamamlanmıştır:

1. ✅ **Veri Yükleme**: Notebook 1'den özellik matrisi yüklendi
2. ✅ **Veri Bölme**: Stratified 70/15/15 split
3. ✅ **Boyut İndirgeme**: PCA (%95 varyans) ve LDA
4. ✅ **10 Model Eğitimi**: SVM (3 kernel), RF, XGBoost, LightGBM, CatBoost, KNN, LR, NB
5. ✅ **Hiperparametre Optimizasyonu**: GridSearchCV ve Optuna
6. ✅ **Çapraz Doğrulama**: Stratified 5-Fold CV
7. ✅ **Model Karşılaştırma**: Tablo ve görsel
8. ✅ **Kayıt**: Modeller ve tahmin olasılıkları kaydedildi

**Sonraki Adım:** `03_cnn_transfer_learning.ipynb` notebook'unda derin öğrenme modelleri eğitilecektir.